# 使用 so(3) 相对增量作为手内操作任务指令

## 背景
目前个人和IsaacLab官方对手内操作的任务指令为物体的目标绝对位姿（位置 + 朝向）。

IsaacLab官方在 `InHandReOrientationCommand` 中实现了基于目标朝向的指令采样和更新逻辑，以及基于朝向误差的奖励函数 `track_orientation_inv_l2`。

```python

    # IsaacLab/source/isaaclab_tasks/isaaclab_tasks/manager_based/manipulation/inhand/mdp/commands/orientation_command.py

    def _resample_command(self, env_ids: Sequence[int]):
        # sample new orientation targets
        rand_floats = 2.0 * torch.rand((len(env_ids), 2), device=self.device) - 1.0
        # rotate randomly about x-axis and then y-axis
        quat = math_utils.quat_mul(
            math_utils.quat_from_angle_axis(rand_floats[:, 0] * torch.pi, self._X_UNIT_VEC[env_ids]),
            math_utils.quat_from_angle_axis(rand_floats[:, 1] * torch.pi, self._Y_UNIT_VEC[env_ids]),
        )
        # make sure the quaternion real-part is always positive
        self.quat_command_w[env_ids] = math_utils.quat_unique(quat) if self.cfg.make_quat_unique else quat

    def _update_command(self):
        # update the command if goal is reached
        if self.cfg.update_goal_on_success:
            # compute the goal resets
            goal_resets = self.metrics["orientation_error"] < self.cfg.orientation_success_threshold
            goal_reset_ids = goal_resets.nonzero(as_tuple=False).squeeze(-1)
            # resample the goals
            self._resample(goal_reset_ids)

    # IsaacLab/source/isaaclab_tasks/isaaclab_tasks/manager_based/manipulation/inhand/mdp/rewards.py

    def track_orientation_inv_l2(
        env: ManagerBasedRLEnv,
        command_name: str,
        object_cfg: SceneEntityCfg = SceneEntityCfg("object"),
        rot_eps: float = 1e-3,
    ) -> torch.Tensor:
        """Reward for tracking the object orientation using the inverse of the orientation error.

        The reward is the inverse of the orientation error between the object orientation and the goal orientation.

        Args:
            env: The environment object.
            command_name: The command term to be used for extracting the goal.
            object_cfg: The configuration for the scene entity. Default is "object".
            rot_eps: The threshold for the orientation error. Default is 1e-3.
        """
        # extract useful elements
        asset: RigidObject = env.scene[object_cfg.name]
        command_term: InHandReOrientationCommand = env.command_manager.get_term(command_name)

        # obtain the goal orientation
        goal_quat_w = command_term.command[:, 3:7]
        # calculate the orientation error
        dtheta = math_utils.quat_error_magnitude(asset.data.root_quat_w, goal_quat_w) # 轴角表示，L2范数

        return 1.0 / (dtheta + rot_eps)

    # IsaacLab/source/isaaclab_tasks/isaaclab_tasks/manager_based/manipulation/inhand/inhand_env_cfg.py
        # -- command terms
        goal_pose = ObsTerm(func=mdp.generated_commands, params={"command_name": "object_pose"})
        goal_quat_diff = ObsTerm(
            func=mdp.goal_quat_diff,
            params={"asset_cfg": SceneEntityCfg("object"), "command_name": "object_pose", "make_quat_unique": False},
        )
```

从这里可以看出，IsaacLab官方的任务是随机重定向，通过绕x轴/y轴的复合运动的随机采样一个绝对目标位姿 $R_g$，来作为目标，并配有当前姿态距离目标姿态的姿态误差 $R_{err}$，两者共同作为观察空间的一部分。

个人当初的实现，则是基于手内旋转任务，通过在当前物体姿态 $R_c$ 的基础上，采样一个相对增量 $\Delta R$（和旋转轴的选取有关），来得到目标姿态 $R_g = R_c \Delta R$,同样是 $R_g$ 和 $R_{err}$ 作为观察空间的一部分。

```python
# AnyMani/source/anymani/anymani/tasks/inhand/mdp/commands/rotation_command.py
    def _resample_command(self, env_ids: Sequence[int]):
        """重置命令并采样新的初始目标姿态。"""

        if len(env_ids) == 0:
            return

        axis_key = self.cfg.rotation_axis.lower()
        if axis_key not in _AXIS_MAP:
            raise ValueError(
                f"不支持的旋转轴 '{self.cfg.rotation_axis}'. 支持项为: {sorted(set(_AXIS_MAP.keys()))}."
            )

        axis_vec = _AXIS_MAP[axis_key].to(self.device)
        self.rotation_axis_w[env_ids] = axis_vec

        # 在物体当前姿态基础上叠加 delta_angle，作为重置后的初始目标姿态
        # 这样重置后的第一个目标与后续目标保持一致的角度增量
        axis_batch = axis_vec.repeat(len(env_ids), 1)
        delta_quat = math_utils.quat_from_angle_axis(self.delta_angle[env_ids], axis_batch)
        base_quat = self.object.data.root_quat_w[env_ids]
        self.quat_command_w[env_ids] = math_utils.quat_mul(delta_quat, base_quat)
        if self.cfg.make_quat_unique:
            self.quat_command_w[env_ids] = math_utils.quat_unique(self.quat_command_w[env_ids])

        self.cumulative_rotation[env_ids] = 0.0
        self.success_counter[env_ids] = 0.0
        self.metrics["orientation_error"][env_ids] = 0.0
        self.metrics["cumulative_rotation"][env_ids] = 0.0
        self.metrics["consecutive_success"][env_ids] = 0.0

    def _update_command(self):
        """根据成功判定沿固定轴增量旋转目标姿态。"""

        if not self.cfg.update_goal_on_success:
            return

        success_mask = self.metrics["orientation_error"] < self.cfg.orientation_success_threshold
        success_ids = success_mask.nonzero(as_tuple=False).squeeze(-1)
        if len(success_ids) == 0:
            return

        # 成功后沿同一轴推进固定角度，形成连续的目标序列
        delta = torch.full((len(success_ids),), self.cfg.delta_angle, device=self.device)
        delta_quat = math_utils.quat_from_angle_axis(delta, self.rotation_axis_w[success_ids])
        updated = math_utils.quat_mul(delta_quat, self.quat_command_w[success_ids])
        if self.cfg.make_quat_unique:
            updated = math_utils.quat_unique(updated)
        self.quat_command_w[success_ids] = updated

        self.cumulative_rotation[success_ids] += self.cfg.delta_angle
        self.success_counter[success_ids] += 1.0
        self.command_counter[success_ids] += 1
        max_time = self.cfg.resampling_time_range[1]
        self.time_left[success_ids] = max_time

# AnyMani/source/anymani/anymani/tasks/inhand/inhand_env_cfg.py
    # -- command terms
    goal_pose = ObsTerm(
        func=mdp.generated_commands,
        params={"command_name": "goal_pose"},
    )
    goal_quat_diff = ObsTerm(
        func=leap_mdp.goal_quat_diff,
        params={
            "asset_cfg": SceneEntityCfg("object"),
            "command_name": "goal_pose",
            "make_quat_unique": True,
        },
    )
```

## 问题
通过以上实例，这便带来了两个问题。
一个是绝对目标位姿的任务指令作为观察空间，在实机部署不便，因为实机中无法直接获取物体的绝对位姿。
二是该种类型的指令通用性不够，个人目前的实现，训练出的策略，要么是绕x轴，要么是绕z轴，缺乏灵活性和泛化能力。

## 新方案
对此，想到了一个新的方案，系统性表述如下。

采用 $so(3)$ 作为相对增量指令，即随机采样一个旋转向量 $\phi$ 作为指令，表示物体当前姿态 $R_c$ 需要绕某一轴旋转 $\phi$ 弧度，旋转轴为 $\frac{\phi}{\left| \phi \right|}$ 的单位向量，从而得到目标姿态 $R_g = \exp(\phi) R_c$。

$so(3)$ 是李代数，$\mathcal{S}^2$ 上的均匀分布采样是能做到的。设 $\phi = u \theta = (u_x,u_y,u_z) \theta$，其中 $(u_x,u_y,u_z)$ 是单位向量，$\theta$ 是旋转角度，那么
$$x,y,z \sim \mathcal{N}(0,1),\quad u = \frac{(x,y,z)}{\sqrt{x^2+y^2+z^2}},\quad \theta \sim U(\theta_{min}, \theta_{max})$$
通常取 $\theta_{min}=0, \theta_{max}=\frac{\pi}{2}$，但 $\theta_{max}$ 不能取 $\pi$，为规避奇异的影响。

在旋转过程中，$\phi_{err} = log(R_g R_c^{-1})$。观察空间中的指令，是 $\phi_{err}$，而非绝对目标位姿 $R_g$ 或 $\phi$。关联的奖励项设计也是围绕 $\phi_{err}$ 进行,这个和 `IsaacLab/source/isaaclab_tasks/isaaclab_tasks/manager_based/manipulation/inhand/mdp/rewards.py` 中的基本一致。在仿真训练中，我们给出的指令是随机的，意味着训练出的策略是可围绕任意轴向的重定向运动，而更规律、绕固定轴的手内旋转（如z轴）可由连续的重定向复合而成，这样就保证了手内操作策略的通用性。在实机部署中，虽无法直接获取物体的绝对位姿（也没必要，不同物体的初始位姿需事先规定，才能确定各种状态下的位姿），但通过设置固定的 $\phi_{err}$ 指令，同样可以鼓励实现连续的手内旋转，不同的 $\phi_{err}$ 的选定，可能会和手内旋转速度相关。而设置线性递减至 0 的 $\phi_{err}$ 指令，则可以实现将物体旋转至某一姿态的任务，只是无法精确对齐某一角度，但也没必要。推理和部署阶段，通过 $\phi_{err}$ 的设计，或许可以呈现出不同的手内操作模式。

这里有一个值得注意的地方—坐标系约定：我们手内旋转，不是绕物体的坐标系 $\{o\}$ 或世界坐标系 $\{w\}$，而是绕手掌自身的的坐标系 $\{s\}$。按照约定，掌心朝外为z轴，拇指为x轴，其余指为y轴。如果手掌朝不同的方向，亦是如此，这样才能保证 $\phi_{err}$ 语义的一致性。
> 不同手的urdf或usd模型中，$\{s\}$ 的定义可能不一样。但可以通过旋转变换等价，总之进行策略训练，验证方案的可行性和效果，再来考虑工程上的统一问题

### 其他需求
**可视化**<br>
和 `IsaacLab/source/isaaclab_tasks/isaaclab_tasks/manager_based/manipulation/inhand/mdp/commands/orientation_command.py` 类似。

**指标**<br>
需实现 `_update_metrics` 方法。有一个未来的长远需求：模仿学习，即 RL 训练手内操作策略 $\pi_\theta$ 用来蒸馏数据集，自然是需要成功，而不是物体被卡住、跌落的数据。这个时候，设计一些实用性的指标帮助过滤轨迹就很有必要了。我们蒸馏的数据，通常是在推理阶段采集的，而且是在 `episode_length_s` 时间内统计指标的，可以通过这些指标，来判断轨迹的质量。

```python
# IsaacLab/source/isaaclab_tasks/isaaclab_tasks/manager_based/manipulation/inhand/mdp/commands/orientation_command.py
        # -- compute the orientation error
        self.metrics["orientation_error"] = math_utils.quat_error_magnitude(
            self.object.data.root_quat_w, self.quat_command_w
        )
        # -- compute the position error
        self.metrics["position_error"] = torch.norm(self.object.data.root_pos_w - self.pos_command_w, dim=1)
        # -- compute the number of consecutive successes
        successes = self.metrics["orientation_error"] < self.cfg.orientation_success_threshold
        self.metrics["consecutive_success"] += successes.float()
```
首先这些指标可以保留，但还可以新增一些，比如累计旋转角度 `cumulative_rotation`，表示物体在整个 episode 中旋转的总角度，单位是弧度。这个指标可以帮助我们判断物体是否进行了足够的旋转操作，从而过滤掉那些几乎没有动弹的轨迹。还有 `cumulative_time`，如果和设定的 `episode_length_s` 差不多，说明应该没有出现跌落等重置事件，要是远小于，说明物体可能从手中跌落了，回合不完整。

## 实验记录
### `AnyMani/logs/rl_games/leaphand_object_rot/2026-02-04_21-16-42`
TODO：训练效果非常差劲，需要排查

## 附录 A：为什么高斯采样归一化后是球面均匀分布？(Generation form AI)
### **结论先行**

$\mathbf{u} = \frac{(X,Y,Z)}{\sqrt{X^2+Y^2+Z^2}}$ 服从**单位球面 $S^2$ 上的均匀分布**（uniform distribution on the unit sphere）。

### **解析推导**

#### **方法一：利用高斯分布的旋转不变性**

> **核心性质**：多元标准正态分布具有球对称性（spherical symmetry）

**步骤1：联合密度函数**

由于 $X, Y, Z \sim \mathcal{N}(0,1)$ 独立，联合密度为：

$$
f_{X,Y,Z}(x,y,z) = \frac{1}{(2\pi)^{3/2}} e^{-\frac{x^2+y^2+z^2}{2}}
$$

**关键观察**：这个密度**只依赖于** $r^2 = x^2+y^2+z^2$，与方向无关！

$$
f_{X,Y,Z}(x,y,z) = \frac{1}{(2\pi)^{3/2}} e^{-\frac{r^2}{2}} = g(r)
$$

**步骤2：球坐标变换**

引入球坐标 $(r, \theta, \phi)$：

$$
\begin{cases}
X = r \sin\theta \cos\phi \\
Y = r \sin\theta \sin\phi \\
Z = r \cos\theta
\end{cases}
$$

其中：
- $r \in [0, \infty)$ 是径向距离
- $\theta \in [0, \pi]$ 是极角（与z轴夹角）
- $\phi \in [0, 2\pi)$ 是方位角

**Jacobian 行列式**：

$$
J = r^2 \sin\theta
$$

**步骤3：变换后的联合密度**

$$
f_{R,\Theta,\Phi}(r,\theta,\phi) = \frac{1}{(2\pi)^{3/2}} e^{-\frac{r^2}{2}} \cdot r^2 \sin\theta
$$

**关键分解**：

$$
f_{R,\Theta,\Phi}(r,\theta,\phi) = \underbrace{\left[\frac{1}{\sqrt{2\pi}} r^2 e^{-\frac{r^2}{2}}\right]}_{f_R(r)} \cdot \underbrace{\left[\frac{\sin\theta}{4\pi}\right]}_{f_{\Theta,\Phi}(\theta,\phi)}
$$

这表明 **$R$ 与 $(\Theta, \Phi)$ 相互独立**！

**步骤4：方向的分布**

$(\Theta, \Phi)$ 的联合密度为：

$$
f_{\Theta,\Phi}(\theta,\phi) = \frac{\sin\theta}{4\pi}, \quad \theta \in [0,\pi], \phi \in [0,2\pi)
$$

这正是**球面均匀分布**的密度！因为球面面积元素为：

$$
dS = \sin\theta \, d\theta \, d\phi
$$

而总面积为 $4\pi$，所以均匀分布的密度为 $\frac{1}{4\pi} \sin\theta$。

#### **方法二：利用对称性的直观论证**

**引理**：对于任意正交矩阵 $Q \in SO(3)$，有

$$
Q \begin{pmatrix} X \\ Y \\ Z \end{pmatrix} \sim \mathcal{N}(\mathbf{0}, I_3)
$$

即旋转后的向量仍然服从相同的分布。

**推论**：归一化向量 $\mathbf{u}$ 的分布必须在所有旋转下保持不变，因此只能是球面均匀分布。

### **附加：径向分布**

顺便，我们也得到了径向距离 $R = \sqrt{X^2+Y^2+Z^2}$ 的分布：

$$
f_R(r) = \frac{1}{\sqrt{2\pi}} r^2 e^{-\frac{r^2}{2}}, \quad r \geq 0
$$

这是**卡方分布** $\chi_3$ 的平方根，即 $R^2 \sim \chi^2(3)$。

### **推广到 $n$ 维**

这个结果可以推广到任意维度：

> **定理**：如果 $\mathbf{X} = (X_1, \ldots, X_n) \sim \mathcal{N}(\mathbf{0}, I_n)$，则
> $$
> \mathbf{U} = \frac{\mathbf{X}}{\|\mathbf{X}\|} \sim \text{Uniform}(S^{n-1})
> $$
> 其中 $S^{n-1}$ 是 $n$ 维单位球面。

这就是为什么**高斯采样 + 归一化**是生成球面均匀分布的标准方法！